🧩 Bloque 1 – Importación de librerías y configuraciones

En este bloque se importan las librerías necesarias para:
- Manipular archivos Excel y datos con pandas y numpy.
- Conectarse a la base de datos SQL Server mediante SQLAlchemy.
- Trabajar con fechas, rutas y expresiones regulares.
    
Estas herramientas permiten la automatización del proceso completo de lectura, limpieza y carga de datos.

In [ ]:
import pandas as pd

import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR = (PROJECT_ROOT / "config" / "CCLA").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    # Mensaje más útil al depurar imports
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"sys.path{sys.path[0] if sys.path else '<vacío>'}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_conosur.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    # 'data' es un dict, no tiene __file__. Imprime la ruta del archivo real.
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()) if isinstance(data, dict) else type(data))

except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
FUNCTIONS_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\functions exists: True
CONFIG_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\config exists: True
Módulo importado desde: C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py
Config cargada desde: C:\Users\aepmvalenzuela\traspaso-innominados\config\config_conosur.json
Claves disponibles: ['tablas_remotas', 'server_config']


⚙️ Bloque 2 – Configuración de conexión a SQL Server

Aquí se definen los parámetros del servidor, base de datos, esquema y tabla destino.

Luego, se crea el motor de conexión con SQLAlchemy, que será utilizado para insertar los datos procesados en SQL Server.

La conexión usa autenticación de Windows y optimiza la velocidad con fast_executemany.

In [2]:
# --- Parámetros iguales a los tuyos ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

# --- Nuevo: construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

Bloque 3 – Definición de ruta y hoja objetivo

Se indica la carpeta donde se encuentran los archivos Excel.

Se filtran los archivos válidos por extensión (.xlsx).

Se define la hoja preferida a leer (por nombre o índice).

Este bloque prepara el entorno para ubicar correctamente los archivos a procesar.

In [3]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "CONOSUR").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["Cesantia_Conosur", "cesantia_conosur", "conosur_cesantia"]
FALLBACK_SHEET_INDEX = 1

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
RUTA_ARCHIVOS: C:\Users\aepmvalenzuela\traspaso-innominados\data\CCLA\CONOSUR exists: True


📑 Bloque 5 – Detección y carga del archivo más reciente

Busca el archivo Excel más reciente dentro de la carpeta indicada.

Si el archivo está bloqueado (por OneDrive, etc.), se crea una copia temporal para poder leerlo.

Detecta automáticamente la hoja objetivo según los nombres configurados.

Así, el script siempre trabaja con el archivo actualizado más reciente.

In [4]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen   = info["archivo_origen"]
excel_path_used  = info["excel_path_used"]
_tmp_copy_path   = info["tmp_copy_path"]
target           = info["target_sheet"]
sheet_names      = info["sheet_names"]

# (Opcional) leer hoja
df_opexcel = fun.read_excel_safe_no_header(excel_path_used, target)
fun.pretty_table(df_opexcel,n=5)

Archivo más reciente: 10_2025_Facturación_Cesantía.xlsx  |  2025-12-26 13:03:00
Hojas: ['resumen', 'Cesantia_Conosur']
Hoja objetivo: Cesantia_Conosur


Bloque 6 – Normalización de encabezados

Se toma la primera fila como encabezado.

Se normalizan y limpian los nombres de columnas usando las funciones del bloque anterior.

Se garantiza que todos los encabezados sean únicos para evitar conflictos.

Esto genera un DataFrame estructurado y fácil de manipular.

In [5]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw0 = fun.read_excel_safe_no_header(io, target)

# Elimina hasta 2 filas iniciales si están mayormente vacías
df_raw0, _removed_preview = fun.drop_initial_empty_rows(
    df_raw0, max_check_rows=2, empty_threshold=0.8, verbose=True
)

if df_raw0.empty:
    raise SystemExit("El archivo quedó sin datos tras eliminar filas iniciales.")

# Tomar fila 0 como encabezado (string)
header_row = df_raw0.iloc[0].astype(str).fillna("").tolist()

# Normalizar nombres de columnas
norm_cols = [
    fun.normalize_name(x) if fun.normalize_name(x) else f"col_{i+1}"
    for i, x in enumerate(header_row)
]

# Hacer únicos (asumes que ya tienes fun.make_unique)
norm_cols = fun.make_unique(norm_cols)

# Construir DF final
df_raw = df_raw0.iloc[1:].reset_index(drop=True).copy()
df_raw.columns = norm_cols

# Revisar duplicados (por seguridad extra)
dups = [c for c in df_raw.columns if df_raw.columns.tolist().count(c) > 1]
if dups:
    print("Aún hay duplicados inesperados en columnas:", sorted(set(dups)))


🧹 Fila inicial 0 eliminada (92% vacía).
🧹 Fila inicial 1 eliminada (92% vacía).


✂️ Bloque 8 – Limpieza de datos de texto

Se eliminan espacios y valores como "None", "NaN", etc.

Se estandarizan los textos para evitar errores en conversiones posteriores.

Permite mantener la consistencia en columnas de tipo texto antes del mapeo.

In [6]:
obj_cols = df_raw.select_dtypes(include=["object", "string"]).columns
if len(obj_cols) > 0:
    df_raw[obj_cols] = (
        df_raw[obj_cols]
        .apply(lambda s: s.astype(str).str.strip())
        .replace({"None": "", "none": "", "NaN": "", "nan": ""})
    )

print("Encabezados normalizados:", list(df_raw.columns))

fun.pretty_table(df_raw, n=5)

Encabezados normalizados: ['foliocredito', 'rutafiliado', 'dvafiliado', 'nombreafiliado', 'plazo', 'montobruto', 'fecotorgamiento', 'fechaprimervto', 'fechaultimovto', 'valorcuota', 'fechaprima', 'prima', 'desgravamen', 'fechadefuncion', 'origendefuncion', 'producto', 'folioorigen', 'fechaorigen', 'tasaorigen', 'poliza', 'tasa', 'prima_bruta_mensual', 'iva', 'nan', 'nan_2']


Bloque 8.1 – Normalización de encabezados

Parcha para buscar las ultimas filas, y ver si son algun TOTAL o PROMEDIO.

De ser asi, se eliminan

In [7]:
df_raw = fun.drop_trailing_mostly_null(
    df_raw,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True
)

Fila índice 3385 → null_ratio=92.00%
Fila índice 3384 → null_ratio=4.00%

🧹 Eliminando 1 fila(s) finales mayoritariamente nulas:

--- Fila eliminada (índice original 3385) ---
     foliocredito rutafiliado dvafiliado nombreafiliado plazo   montobruto  \
3385                                                           11479768584   

     fecotorgamiento fechaprimervto fechaultimovto          valorcuota  ...  \
3385                                                127445.45435745938  ...   

     producto folioorigen fechaorigen tasaorigen poliza tasa  \
3385                                                           

     prima_bruta_mensual iva nan nan_2  
3385                               NaN  

[1 rows x 25 columns]




🧭 Bloque 9 – Mapeado de columnas al formato estándar

Se buscan las columnas relevantes del archivo original, aunque tengan distintos nombres.

Se crea un nuevo DataFrame (df_can) con nombres normalizados y uniformes.

Este paso alinea la estructura del Excel con la esperada por la base de datos SQL.

In [8]:
# Origen → Destino
foliocredito        = fun.pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
rutafiliado         = fun.pick(df_raw, "afirut", "rut_afiliado", "rut", "rutafiliado")
dvafiliado          = fun.pick(df_raw, "afirutdv", "dv_afiliado", "dv", "dvafiliado")
NombreAfiliado      = fun.pick(df_raw, "afinom", "nombre_afiliado", "nombre", "nombreafiliado")
Plazo               = fun.pick(df_raw, "crecuotot", "plazo")
MontoBruto          = fun.pick(df_raw, "cresolmon", "monto_bruto", "monto", "montobruto")
fecotorgamiento     = fun.pick(df_raw, "fecotorgamiento", "fecha_otorgamiento", "fec_otorgamiento")
fechaPrimerVto      = fun.pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob", "fechaprimervto")
FechaUltimoVto      = fun.pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob", "fechaultimovto")
ValorCuota          = fun.pick(df_raw, "crecuomon", "valor_cuota", "valorcuota")
FechaPrima          = fun.pick(df_raw, "fechaprima", "fecha_prima")
Prima               = fun.pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_" , "prima")
Desgravamen         = fun.pick(df_raw, "prima_desgravamen", "desgravamen")
FechaDefuncion      = fun.pick(df_raw, "fecdefuncion", "fecha_defuncion", "fechadefuncion")
OrigenDefuncion     = fun.pick(df_raw, "origen_defuncion", "origin_defuncion", "origendefuncion")
Producto            = fun.pick(df_raw, "producto")
FolioOrigen         = fun.pick(df_raw, "folio_original", "folio_origen", "folioorigen")
FechaOrigen         = fun.pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen", "fechaorigen")
TasaOrigen          = fun.pick(df_raw, "tasa_interes", "tasaorigen")
Poliza              = fun.pick(df_raw, "poliza", "póliza")
Tasa                = fun.pick(df_raw, "tasa_interes", "tasa")
PrimaBruta          = fun.pick(df_raw, "prima_bruta_mensual", "prima_bruta")
Iva                 = fun.pick(df_raw, "iva")
PrimaNeta           = fun.pick(df_raw, "primaneta", "prima_neta", "nan")


df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "NombreAfiliado": NombreAfiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fecotorgamiento": fecotorgamiento,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "FechaPrima": FechaPrima,
    "Prima": Prima,
    "Desgravamen": Desgravamen,
    "FechaDefuncion": FechaDefuncion,
    "OrigenDefuncion": OrigenDefuncion,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "TasaOrigen": TasaOrigen,
    "POLIZA": Poliza,
    "TASA": Tasa,
    "PrimaBrutaMensual": PrimaBruta,
    "IVA": Iva,
    "PrimaNetaMensual": PrimaNeta,
    "Nombre_de_archivo": archivo_origen.name,
})

fun.pretty_table(df_can, n=5)

🗂️ Bloque 10 – Normalización del nombre del archivo

Se extrae automáticamente el período (YYYYMM) desde el nombre del archivo.

Si no se encuentra, se genera un nombre genérico con la fecha actual.

El resultado se guarda en la columna Nombre_de_archivo, clave para la trazabilidad en SQL.

In [9]:
nombre_canonico = fun.canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : 10_2025_Facturación_Cesantía.xlsx
Nombre canónico : Facturacion_Cesantia_202510.xlsx


🔢 Bloque 11 – Conversión y tipificación de columnas

Se convierten columnas al tipo correcto (entero, flotante o texto).

Se limpian valores nulos, se formatean mayúsculas y se ajusta la longitud de texto.

Este paso asegura compatibilidad entre los datos procesados y la estructura SQL.

In [10]:
# 1) Configuración de columnas
INT_COLS    = [
    "rutafiliado", "Plazo", "fecotorgamiento", "fechaPrimerVto", "FechaUltimoVto", "FechaPrima",
    "FechaDefuncion", "FechaOrigen", "POLIZA"
]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["MontoBruto", "ValorCuota", "Prima", "Desgravamen", "TasaOrigen", "TASA",
               "PrimaBrutaMensual", "IVA", "PrimaNetaMensual"]
STR_COLS    = ["NombreAfiliado", "OrigenDefuncion", "Producto", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"

# 2) Aplicación de castings/limpiezas sobre df_can
#    (asumo que df_can ya existe en tu contexto)
df_can = fun.cast_numeric_columns(
    df_can,
    bigint_cols=BIGINT_COLS,
    int_cols=INT_COLS,
    float_cols=FLOAT_COLS
)

df_can = fun.normalize_dv_column(df_can, DV_COL)

# 3) Recortes de strings (límites según tu lógica)
df_can = fun.trim_string_columns(
    df_can,
    limits={
        "NombreAfiliado": 255,
        "OrigenDefuncion": 255,
        "Producto": 255,
        "Nombre_de_archivo": 50,
    }
)

print("dtypes DESPUÉS:\n", df_can.dtypes)

# 4) Reporte nulos en columnas críticas
criticas = ["foliocredito", "rutafiliado", "dvafiliado", "Nombre_de_archivo"]
fun.report_nulls(df_can, criticas)

# 5) Armado de df_sql con orden fijo (solo columnas presentes)
cols_sql = [
    "foliocredito","rutafiliado","dvafiliado","NombreAfiliado","Plazo","MontoBruto","fecotorgamiento",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","FechaPrima","Prima","Desgravamen","FechaDefuncion",
    "OrigenDefuncion","Producto","FolioOrigen", "FechaOrigen","TasaOrigen","POLIZA","TASA","PrimaBrutaMensual",
    "IVA","PrimaNetaMensual","Nombre_de_archivo"
]

df_sql = fun.build_sql_frame(df_can, cols_sql)

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

dtypes DESPUÉS:
 foliocredito                  Int64
rutafiliado                   Int64
dvafiliado                   object
NombreAfiliado       string[python]
Plazo                         Int64
MontoBruto                  float64
fecotorgamiento               Int64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
FechaPrima                    Int64
Prima                       float64
Desgravamen                 float64
FechaDefuncion                Int64
OrigenDefuncion      string[python]
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
TasaOrigen                  float64
POLIZA                        Int64
TASA                        float64
PrimaBrutaMensual           float64
IVA                         float64
PrimaNetaMensual            float64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliad

🔍 Bloque 12 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [11]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Facturacion_Cesantia_202510.xlsx: 3385 filas


🏁 Bloque 13 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [12]:
resumen = []

# 1) Resolver schema/table por si viene "dbo.tabla" o solo "tabla"
table_in = data["tablas_remotas"]["conosur_acumulado_r_bkp"]

# Si viene con schema incluido, lo separamos (sin crear funciones)
if "." in table_in:
    sch0, tbl0 = table_in.split(".", 1)
    schema = sch0.strip().strip("[]")
    table = tbl0.strip().strip("[]")
else:
    table = table_in.strip().strip("[]")

full_table = fun._full_table_sqlserver(schema, table)
col_part = fun._qident_sqlserver("Nombre_de_archivo")

for nombre_archivo in vals:
    # 2) Filtrar subset del DF
    # (pandas) -> df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]
    # (polars) -> df_sql.filter(pl.col("Nombre_de_archivo") == nombre_archivo)
    # Como df_to_db detecta el engine, acá asumimos pandas como en tu código original:
    df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

    if df_sub.empty:
        print(f"No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
        continue

    # 3) COUNT en destino con query_to_df (escapando comillas simples)
    nombre_safe = str(nombre_archivo).replace("'", "''")
    sql_count = f"""
        SELECT COUNT(*) AS cnt
        FROM {full_table}
        WHERE {col_part} = '{nombre_safe}'
    """
    df_cnt = fun.query_to_df(sql_count, connection_string=connection_string, engine="pandas")
    existing_count = int(df_cnt["cnt"].iloc[0]) if df_cnt is not None and len(df_cnt) else 0

    # 4) Mensajería similar a la original
    if existing_count > 0:
        print(
            f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
            f"({existing_count} filas en {schema}.{table})."
        )
        print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")
        accion = "reemplazo"
    else:
        print(
            f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
            "Se insertará como archivo nuevo."
        )
        accion = "inserción nueva"


    # 5) Reemplazo real: DELETE (por partición) + INSERT (df_sub) usando df_to_db
    #    Esto reemplaza tu DELETE + to_sql(append)
    summary = fun.df_to_db(
        df_sub,
        connection_string=connection_string,
        schema=schema,
        table=table,
        mode="replace_partition",
        partition_column="Nombre_de_archivo",
        partition_values=[nombre_archivo],
        engine="auto",           # detecta pandas/polars según df_sub
        chunksize=20_000,
        commit_every_chunk=False
    )

    deleted = int(summary.get("rows_deleted", 0) or 0)
    inserted = len(df_sub)  # df_to_db también trae rows_inserted, pero esto es exacto por df_sub

    if existing_count > 0:
        print(f"Filas eliminadas en destino para '{nombre_archivo}': {deleted}")

    print(f"📥 Insertadas {inserted} filas para Nombre_de_archivo = '{nombre_archivo}'.")
    resumen.append((nombre_archivo, inserted, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

♻️ Se encontró data previa para Nombre_de_archivo = 'Facturacion_Cesantia_202510.xlsx' (3385 filas en dbo.Conosur_Acumulado_R_BKP).
🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...
Filas eliminadas en destino para 'Facturacion_Cesantia_202510.xlsx': 3385
📥 Insertadas 3385 filas para Nombre_de_archivo = 'Facturacion_Cesantia_202510.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - Facturacion_Cesantia_202510.xlsx: 3385 filas cargadas (reemplazando 3385 previas).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [13]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\nNo se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\nNo se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\nError inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepmvalenzuela\traspaso-innominados\data\CCLA\CONOSUR\10_2025_Facturación_Cesantía.xlsx


# SQL

## VALIDACIONES / INSPECCIÓN

In [14]:
query_1 = f"""
-- [CONOSUR] Archivos cargados (desc)
SELECT DISTINCT 
    Nombre_de_archivo
FROM {data["tablas_remotas"]["conosur_acumulado_r_bkp"]}
ORDER BY 
    Nombre_de_archivo desc
"""

df_q1 = fun.query_to_df(
    sql=query_1,
    connection_string=connection_string,
    engine="pandas"
)

fun.pretty_table(df_q1)

- Pseudo-tibble: 73 × 1 Pandas
Nombre_de_archivo <object>
Facturacion_Cesantia_202601.xlsx
Facturacion_Cesantia_202512.xlsx
Facturacion_Cesantia_202511.xlsx
Facturacion_Cesantia_202510.xlsx
Facturacion_Cesantia_202509.xlsx
Facturacion_Cesantia_202508.xlsx
Facturacion_Cesantia_202507.xlsx
Facturacion_Cesantia_202506.xlsx
Facturacion_Cesantia_202505.xlsx


## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

In [15]:
query_2 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["conosur_final_acumulado_bkp"]};
"""

fun.exec_sql(query_2, connection_string)

{'ok': False,
 'error': 'ProgrammingError(\'42S02\', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot drop the table \'CONOSUR_FINAL_ACUMULADO_BKP\', because it does not exist or you do not have permission. (3701) (SQLExecDirectW)")',
 'seconds': 0.056}

In [ ]:
query_3 = f"""
SELECT *,
       /* 6354562 as POLIZA, -- comentado en tu script */
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       6832 as PLAN_TECNICO,
       4 as PLAZO_CUOTAS,
       'Credito Consumo' as Negocio
INTO {data["tablas_remotas"]["conosur_final_acumulado_bkp"]}
FROM {data["tablas_remotas"]["conosur_acumulado_r_bkp"]};
"""

fun.exec_sql(query_3, connection_string)